# VIBE dataset vs a Gaussian with the same spectrum: robust-prune graphs

1. **Download** one dataset (`DATASET`) from [vector-index-bench/vibe](https://huggingface.co/datasets/vector-index-bench/vibe) into `DOWNLOAD_DIR`, using only the Python standard library. The file streams into `<name>.hdf5.part`, an interrupted download resumes, and the finished file is checked against the size and SHA-256 that Hugging Face lists. A complete file already in `DOWNLOAD_DIR` is reused. This cell needs internet access.
2. **Spectrum**: the exact covariance spectrum and effective dimension $d_\mathrm{eff}$ of the corpus (`train`). A **synthetic Gaussian** gets the same number of points and the same covariance eigenvalues: $x = z \odot \sqrt{\lambda}$, $z \sim \mathcal N(0, I)$. Robust prune under Euclidean distance does not change under rotation or translation, so the eigenvectors and mean are not needed. The eigenvalues are saved; later runs build the Gaussian from the saved file, so it is identical on every run and machine.
3. **Robust prune** (`batchedEuclideanRobustPrune` from `utils.py`; `PRUNE_ALPHA = 1` gives a fully navigable graph) on both datasets, written as adjacency lists in `simulrun.py`'s format, one line per source: `source [neighbor1, neighbor2, ...]`.
4. **Graph statistics** for both, read back from the stored graphs, next to the spectrum statistics.

The code is in `vibe.py`, shared with `vibe_pipeline.py` (the same pipeline as a script, for SLURM). Both write the same files to `GRAPH_DIR`, so with the same settings this notebook **picks up** what the script computed: the download and finished graphs are skipped, and only the spectrum (one pass over the data) and the Gaussian are recomputed for the plots. A partly built graph continues from where it stopped.

**Distances.** The prune is Euclidean: `cosine` / `angular` datasets are L2-normalised first (giving exactly the cosine graph), `any` datasets are already unit-norm, `euclidean` is used as is. `ip`, `hamming` / `binary` and `chamfer` datasets are rejected.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from paths import RESULTS_DIR
from vibe import (build_graph, cuda_available, dataset_spectrum, describe, download, gaussian_like,
                  load_prune, load_train, sample_spectrum, summarize)

DATASET = "landmark-nomic-768-normalized"   # file name without .hdf5 (see the README tables in the repo)
DOWNLOAD_DIR = None      # where to save the .hdf5, e.g. "/scratch/<netid>/vibe"
GRAPH_DIR = RESULTS_DIR / "vibe_graphs"     # graphs, spectrum and results (same default as vibe_pipeline.py)
MAX_POINTS = None        # None: the whole train set; int k: a seeded random subset of k points
SEED = 0
PRUNE_ALPHA = 1.0        # alpha-reachability; 1.0 = basic robust prune (fully navigable graph)
USE_GPU = None           # None: CuPy if it imports and sees a CUDA GPU, else NumPy
BATCH_SIZE = None        # None: 512 on GPU, 64 on CPU. GPU/RAM use ~ 24 * BATCH_SIZE * n bytes
VERIFY_SHA256 = True     # check the download's checksum (reads the file once more, ~seconds per GB)
RECIPROCITY = True       # share of edges u->v whose reverse v->u exists (~24 bytes per edge)

assert DOWNLOAD_DIR is not None, "set DOWNLOAD_DIR to where the dataset should be downloaded"
if USE_GPU is None:
    USE_GPU = cuda_available()
if BATCH_SIZE is None:
    BATCH_SIZE = 512 if USE_GPU else 64
print(f"USE_GPU = {USE_GPU}, BATCH_SIZE = {BATCH_SIZE}  (python: {sys.executable})")

## 1. Download

In [ ]:
path = download(DATASET, DOWNLOAD_DIR, verify=VERIFY_SHA256)
attrs, contents = describe(path)
print(path, f"({path.stat().st_size / 1e9:.2f} GB)")
print("attributes:", attrs)
print("contents:  ", contents)

## 2. Spectrum and the synthetic Gaussian

In [ ]:
X, NAME, _ = load_train(path, MAX_POINTS, SEED)
N, D = X.shape
print(f"{NAME}: {N:,} x {D} float32 ({X.nbytes / 1e9:.2f} GB)")

lam, d_eff = dataset_spectrum(X, NAME, GRAPH_DIR)
X_syn = gaussian_like(lam, N, SEED)
lam_syn, d_eff_syn = sample_spectrum(X_syn, "gaussian")
print(f"d_eff = {d_eff:.1f}  (d_eff / d = {d_eff / D:.3f});  Gaussian sample d_eff = {d_eff_syn:.1f}")

In [ ]:
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834"]   # dataset, synthetic
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
})

# Eigenvalues in decreasing order, divided by the largest.
plt.figure(figsize=(8, 4))
k = np.arange(1, D + 1)
plt.plot(k, lam / lam[0], color=SERIES[0], lw=2, label=DATASET)
plt.plot(k, lam_syn / lam_syn[0], color=SERIES[1], lw=2, ls="--", label="Gaussian (sample spectrum)")
plt.xlim(1, D)
plt.ylim(0, 1)
plt.xlabel("rank k")
plt.ylabel("λ_k / λ_1")
plt.title(f"Covariance spectrum, d_eff = {d_eff:.1f} of d = {D}")
plt.legend(frameon=False)
plt.show()

## 3. Robust prune

Finished graphs (every source in the `-computed.txt` file) are skipped; a partial one continues.

In [ ]:
prune_fns = load_prune(USE_GPU)
print(f"prune: {prune_fns[0].__name__}  (batch size {BATCH_SIZE})")
GRAPHS = {
    DATASET: build_graph(X, NAME, GRAPH_DIR, prune_fns, PRUNE_ALPHA, BATCH_SIZE, SEED),
    "gaussian": build_graph(X_syn, f"{NAME}-gaussian", GRAPH_DIR, prune_fns, PRUNE_ALPHA, BATCH_SIZE, SEED),
}

## 4. Graph statistics

In-degree and reciprocity are reported only for complete graphs. Saved to `GRAPH_DIR/<name>-results.csv`.

In [ ]:
results, out_degrees = summarize(NAME, GRAPHS, {DATASET: (lam, d_eff), "gaussian": (lam_syn, d_eff_syn)},
                                 N, D, GRAPH_DIR, RECIPROCITY)
results.style.format({"n": "{:,}", "d_eff": "{:.1f}", "d_eff / d": "{:.3f}", "sources": "{:,}", "edges": "{:,}",
                      "out mean": "{:.1f}", "out median": "{:.0f}", "out p99": "{:.0f}", "out max": "{:.0f}",
                      "in median": "{:.0f}", "in p99": "{:.0f}", "in max": "{:.0f}", "in zero": "{:,.0f}",
                      "reciprocity": "{:.3f}"}, na_rep="–").hide(axis="index")

In [ ]:
# Out-degree distributions on shared bins.
plt.figure(figsize=(8, 4))
hi = max(x.max() for x in out_degrees.values())
bins = np.arange(0, hi + 2) if hi <= 200 else np.linspace(0, hi + 1, 101)
for color, (label, x) in zip(SERIES, out_degrees.items()):
    counts, edges = np.histogram(x, bins=bins)
    plt.stairs(counts / len(x), edges, color=color, lw=2, label=f"{label} (mean {x.mean():.1f})")
plt.xlabel("out-degree")
plt.ylabel("fraction of sources")
plt.title(f"Robust-prune out-degree, n = {N:,}")
plt.legend(frameon=False)
plt.show()